# Real-data validation: FBP3D ToMoBAR on I12 dataset 1

This case performs FBP3D ToMoBAR reconstruction with dark/flat correction, negative logarithm,
and padded filtered backprojection. It combines numerical regression checks with
images for scientific review.

## Data and execution

Use `i12_dataset1.npz` from [Zenodo record 17252190](https://zenodo.org/records/17252190).
The repository downloader verifies the archive checksums.
Run from the repository root:

```bash
python .scripts/download_zenodo.py Zenodo
python -m pip install -e '.[validation]'
python -m pytest --nbmake --nbmake-timeout=1800 --overwrite docs/source/validation/reconstruction/direct/fbp3d_i12_dataset1.ipynb
```

Use an NVIDIA GPU with compatible CuPy and ToMoBAR installed, plus ASTRA for
ASTRA-based reconstruction. Run notebooks serially. Set `HTTOMOLIBGPU_DATA_DIR`
to use an archive stored elsewhere.

Documentation builds render saved outputs without running GPU computations.
Execute with `--overwrite` before building documentation to update the results.


In [ ]:
import gc
import hashlib
import json
import os
import platform
from importlib.metadata import version
from pathlib import Path

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
from numpy.testing import assert_allclose

from httomolibgpu.prep.normalize import dark_flat_field_correction, minus_log
from httomolibgpu.recon.algorithm import FBP3d_tomobar

# Locate the checkout from either the repository root or the notebook directory.
repo_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "httomolibgpu").is_dir()
    ),
    None,
)
data_dir = os.environ.get("HTTOMOLIBGPU_DATA_DIR")
if data_dir is None:
    if repo_root is None:
        raise FileNotFoundError(
            "Set HTTOMOLIBGPU_DATA_DIR to the Zenodo archive directory"
        )
    data_dir = repo_root / "Zenodo"
data_path = Path(data_dir).expanduser() / "i12_dataset1.npz"
if not data_path.is_file():
    raise FileNotFoundError(
        f"Missing {data_path}; run .scripts/download_zenodo.py first"
    )

checksum = hashlib.sha256()
with data_path.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        checksum.update(chunk)
props = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
provenance = {
    "dataset": data_path.name,
    "zenodo_record": "17252190",
    "sha256": checksum.hexdigest(),
    "python": platform.python_version(),
    "packages": {
        name: version(name)
        for name in ("httomolibgpu", "tomobar", "astra-toolbox", "numpy")
    },
    "cupy": cp.__version__,
    "cuda_runtime": cp.cuda.runtime.runtimeGetVersion(),
    "cuda_driver": cp.cuda.runtime.driverGetVersion(),
    "gpu": (
        props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
    ),
}
print(json.dumps(provenance, indent=2))

## Preprocessing

Input axes are `(angle, detector row, detector column)` and stored angles are in
degrees. Use the default dark/flat correction bounds, then apply `minus_log`,
using a fixed centre of rotation of
1253.75 pixels; this case does not estimate it or optimise parameters.


In [ ]:
with np.load(data_path) as archive:
    projdata = cp.asarray(archive["projdata"])
    flats = cp.asarray(archive["flats"])
    darks = cp.asarray(archive["darks"])
    angles = archive["angles"].copy()

data_normalised = dark_flat_field_correction(projdata, flats, darks)
data_normalised = minus_log(data_normalised)
data_normalised = cp.ascontiguousarray(data_normalised[:, 10:20, :])
del projdata, flats, darks
gc.collect()
cp.get_default_memory_pool().free_all_blocks()
assert bool(cp.isfinite(data_normalised).all()), "Non-finite preprocessed data"
print("Preprocessed shape:", data_normalised.shape)

mid_row = data_normalised.shape[1] // 2
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, image, title, ylabel in (
    (axes[0], data_normalised[0].get(), "First corrected projection", "Detector row"),
    (
        axes[1],
        data_normalised[:, mid_row, :].get(),
        "Central-row sinogram",
        "Angle index",
    ),
):
    im = ax.imshow(image, cmap="gray", aspect="auto")
    ax.set(title=title, xlabel="Detector column", ylabel=ylabel)
    fig.colorbar(im, ax=ax, label="Negative log transmission")
plt.show()
plt.close(fig)

## Reconstruction and regression assertions

Use 580 pixels of detector padding, a SINC filter cutoff of 0.35 and
`recon_mask_radius=2.0` (no circular cropping). Output axes are `(Y, detector row, X)`.

In [ ]:
parameters = {
    "center": 1253.75,
    "detector_pad": 580,
    "filter_freq_cutoff": 0.35,
    "recon_mask_radius": 2.0,
}
print("Reconstruction parameters:", parameters)
recon_gpu = FBP3d_tomobar(data_normalised, np.deg2rad(angles), **parameters)
assert recon_gpu.flags.c_contiguous
recon_data = cp.asnumpy(recon_gpu)
del recon_gpu, data_normalised
gc.collect()
cp.get_default_memory_pool().free_all_blocks()

assert recon_data.dtype == np.float32
assert recon_data.shape == (2560, 10, 2560)
assert np.isfinite(recon_data).all(), "Non-finite reconstructed values"
observed_sum = np.sum(recon_data)
print(
    json.dumps(
        {
            "shape": list(recon_data.shape),
            "dtype": str(recon_data.dtype),
            "sum": float(observed_sum),
            "expected_sum": 17852.43359375,
            "absolute_difference": float(abs(observed_sum - 17852.43359375)),
            "minimum": float(np.min(recon_data)),
            "maximum": float(np.max(recon_data)),
        },
        indent=2,
    )
)
# Check the numerical baseline after plotting so failed runs retain visual evidence.

In [ ]:
image = recon_data[:, recon_data.shape[1] // 2, :]
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
limits = {"vmin": -0.001, "vmax": 0.0015, "cmap": "gray"}
im = axes[0].imshow(image, **limits)
axes[0].set(title="Central reconstructed slice", xlabel="X pixel", ylabel="Y pixel")
axes[1].imshow(image[900:1400, 1000:1500], extent=(1000, 1500, 1400, 900), **limits)
axes[1].set(title="Fixed region of interest", xlabel="X pixel", ylabel="Y pixel")
fig.colorbar(im, ax=axes, label="Reconstructed value (library units)")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 3), constrained_layout=True)
ax.plot(image[image.shape[0] // 2, :])
ax.set(
    title="Central horizontal profile", xlabel="X pixel", ylabel="Reconstructed value"
)
ax.grid(alpha=0.3)
plt.show()
plt.close(fig)

In [ ]:
assert_allclose(observed_sum, 17852.43359375, rtol=1e-7, atol=1e-6)
print("All regression checks passed.")

del image, recon_data
gc.collect()
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()